In [1]:
import pandas as pd

df = pd.read_csv('data/movie_genre_dataset.csv')

# Keep only the columns we need
df = df[['Overview', 'Genre']]

print(df.head())

                                            Overview                 Genre
0  Two imprisoned men bond over a number of years...                 Drama
1  An organized crime dynasty's aging patriarch t...          Crime, Drama
2  When the menace known as the Joker wreaks havo...  Action, Crime, Drama
3  The early life and career of Vito Corleone in ...          Crime, Drama
4  A jury holdout attempts to prevent a miscarria...          Crime, Drama


In [2]:
# "Action, Crime, Drama" becomes ['Action', 'Crime', 'Drama']
df = df.dropna()  # remove empty rows
df['Genre'] = df['Genre'].apply(lambda x: [g.strip() for g in x.split(',')])
print(df['Genre'].head())

0                   [Drama]
1            [Crime, Drama]
2    [Action, Crime, Drama]
3            [Crime, Drama]
4            [Crime, Drama]
Name: Genre, dtype: object


In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['Genre'])
print(mlb.classes_)   # shows all genre names, e.g. Action, Comedy, Drama...
print(y[:5])           # shows first 5 rows as 0/1 patterns

['Action' 'Adventure' 'Animation' 'Biography' 'Comedy' 'Crime' 'Drama'
 'Family' 'Fantasy' 'Film-Noir' 'History' 'Horror' 'Music' 'Musical'
 'Mystery' 'Romance' 'Sci-Fi' 'Sport' 'Thriller' 'War' 'Western']
[[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [4]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_overview'] = df['Overview'].apply(clean_text)
print(df['clean_overview'].head())

0    two imprisoned men bond over a number of years...
1    an organized crime dynastys aging patriarch tr...
2    when the menace known as the joker wreaks havo...
3    the early life and career of vito corleone in ...
4    a jury holdout attempts to prevent a miscarria...
Name: clean_overview, dtype: object


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_overview'], y, test_size=0.2, random_state=42
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 800
Testing samples: 200


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("Shape of training data:", X_train_tfidf.shape)

Shape of training data: (800, 10000)


In [7]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

model = OneVsRestClassifier(LinearSVC(class_weight='balanced'))
model.fit(X_train_tfidf, y_train)
print("Model trained!")

Model trained!


In [8]:
from sklearn.metrics import classification_report, f1_score

y_pred = model.predict(X_test_tfidf)
print("F1 score:", f1_score(y_test, y_pred, average='micro'))
print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))

F1 score: 0.49238578680203043
              precision    recall  f1-score   support

      Action       0.68      0.39      0.50        33
   Adventure       0.92      0.23      0.37        48
   Animation       0.00      0.00      0.00        20
   Biography       0.83      0.19      0.30        27
      Comedy       0.44      0.17      0.24        42
       Crime       0.52      0.42      0.46        31
       Drama       0.77      0.92      0.84       145
      Family       0.00      0.00      0.00         9
     Fantasy       0.00      0.00      0.00         9
   Film-Noir       0.00      0.00      0.00         4
     History       0.00      0.00      0.00        13
      Horror       0.00      0.00      0.00         7
       Music       0.00      0.00      0.00        10
     Musical       0.00      0.00      0.00         5
     Mystery       0.43      0.13      0.20        23
     Romance       0.50      0.14      0.22        21
      Sci-Fi       0.50      0.11      0.17        

In [9]:
def predict_genre(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = model.predict(vec)
    return mlb.inverse_transform(pred)[0]

print(predict_genre("A young wizard discovers he has magical powers and must fight an evil dark lord."))

('Action', 'Adventure', 'Fantasy')


In [10]:
import joblib

joblib.dump(model, 'genre_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(mlb, 'label_binarizer.pkl')

print("Model and tools saved successfully!")

Model and tools saved successfully!


In [11]:
all_genres = df['Genre'].explode()
print(all_genres.value_counts())

Genre
Drama        724
Comedy       233
Crime        209
Adventure    196
Action       189
Thriller     137
Romance      125
Biography    109
Mystery       99
Animation     82
Sci-Fi        67
Fantasy       66
History       56
Family        56
War           51
Music         35
Horror        32
Western       20
Film-Noir     19
Sport         19
Musical       17
Name: count, dtype: int64
